# 03 - Özellik Mühendisliği
## Uydu Telemetri Anomali Tespiti

Zaman serisindeki saniyelik gürültüleri ortadan kaldırmak için anomali tespitini olay bazlı hale getirmemiz lazım. 
ESA'nın orijinal özellik matrisi (`dataset.csv`) ile kendi ürettiğimiz sinyaş işleme özelliklerini birleştireceğiz.



In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import json
import sys
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

sys.path.insert(0, '..')
from src.feature_engineer import TelemetryFeatureEngineer

print('Kütüphaneler ve Feature Engineer yüklendi.')


Kütüphaneler ve Feature Engineer yüklendi.


---
## Veri Yükleme


In [12]:
df_segments = pd.read_csv('../data/raw/segments.csv')

df_dataset = pd.read_csv('../data/raw/dataset.csv')

print(f'Zaman Serisi (Segmets) Boyutu: {df_segments.shape}')
print(f'Hedef Olay (Dataset) Boyutu: {df_dataset.shape}')


Zaman Serisi (Segmets) Boyutu: (303493, 8)
Hedef Olay (Dataset) Boyutu: (2123, 23)


---
## Segment Bazlı Özel Özellik Çıkarımı
Her bir olay için sinayl işleme metriklerini hesaplıyoruz.


In [13]:
engineer = TelemetryFeatureEngineer()

df_custom_features = engineer.extract_segment_features(df_segments)

display(df_custom_features.head())


Segment bazlı istatistikler ve sinyal özellikleri çıkarılıyor...


,segment,custom_rms,custom_p2p,custom_crest_factor,custom_zcr,anomaly,channel
0,1,0.000019,0.000092,3.441630,0.035714,1,CADC0872
1,2,0.000026,0.000079,1.545878,0.012579,1,CADC0872
2,3,0.000026,0.000086,1.695852,0.006723,1,CADC0872
3,4,0.000023,0.000105,2.686044,0.018382,1,CADC0872
4,5,0.000025,0.000067,1.675313,0.007782,0,CADC0872


---
## Özellik Birleştirme
ESA'nın sağladığı `mean`, `var`, `skew` gibi istatistiksel özelliklerle, bizim ürettiğimiz `custom_rms`, `custom_zcr` gibi sinyal özelliklerini birleştiriyoruz.


In [14]:
df_custom_features = df_custom_features.drop(columns=['anomaly', 'channel'])

df_final = pd.merge(df_dataset, df_custom_features, on='segment', how='inner')

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_final['channel_id'] = le.fit_transform(df_final['channel'])

print(f"Birleştirilmiş Veri Seti Boyutu: {df_final.shape}")
display(df_final.head(3))


Birleştirilmiş Veri Seti Boyutu: (2123, 28)


,segment,anomaly,train,channel,sampling,duration,len,mean,var,std,...,diff2_var,gaps_squared,len_weighted,var_div_duration,var_div_len,custom_rms,custom_p2p,custom_crest_factor,custom_zcr,channel_id
0,1,1,1,CADC0872,1,279,280,8.533143e-07,3.494283e-10,0.000019,...,2.960666e-10,309,280,1.252431e-12,1.247958e-12,0.000019,0.000092,3.441630,0.035714,0
1,2,1,1,CADC0872,1,476,477,-3.639396e-06,6.476485e-10,0.000025,...,3.004752e-12,644,477,1.360606e-12,1.357754e-12,0.000026,0.000079,1.545878,0.012579,0
2,3,1,1,CADC0872,1,594,595,1.170788e-05,5.592877e-10,0.000024,...,1.029918e-11,772,595,9.415618e-13,9.399794e-13,0.000026,0.000086,1.695852,0.006723,0


---
## Özellik Matrisini ve Kataloğu Kaydetme


In [15]:
df_final.to_parquet('../data/features/segment_features.parquet')

catalog = {
    "total_segments": len(df_final),
    "features_list": list(df_final.columns),
    "method": "Segment-Level Aggregation & Merging"
}
with open('../data/features/feature_catalog.json', 'w', encoding='utf-8') as f:
    json.dump(catalog, f, indent=4, ensure_ascii=False)

print('Özellik matrisi başarıyla kaydedildi.')


Özellik matrisi başarıyla kaydedildi.
